<a href="https://colab.research.google.com/github/Sandyshor/Msc-Group-Project-Team-1/blob/main/Recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# DUBLIN RESIDENTIAL AREA RECOMMENDATION PROTOTYPE
import os
import re
import textwrap
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/Sandyshor/Msc-Group-Project-Team-1.git"
REPO_DIR = "/content/Msc-Group-Project-Team-1"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull -q")

os.chdir(f"{REPO_DIR}/Data")
print(os.listdir())

# 1. LOAD THE DATA
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df


commute = clean_columns(pd.read_excel("TUD G.xlsx"))

amenities = pd.concat(
    [clean_columns(pd.read_csv(f"{a}_Amenities_Counts.csv")) for a in ["D1", "D2", "D3", "D6", "D7", "D8", "D9", "D10"]],
    ignore_index=True,
)

room_data = pd.concat(
    [clean_columns(pd.read_excel(f"{a}.xlsx")) for a in ["D1", "D2", "D3", "D6", "D7", "D8", "D9", "D10"]],
    ignore_index=True,
)
room_data["Room type"] = room_data["Room type"].astype(str).str.strip().str.lower()

rtb = pd.concat(
    [clean_columns(pd.read_csv(f)) for f in ["RTB_Clean.csv", "RTB_Clean1.csv", "RTB_Clean2.csv", "RTB_Clean3.csv"]],
    ignore_index=True,
)

census = clean_columns(pd.read_excel("Dublin_Census_Filled.xlsx"))
ethnicity = clean_columns(pd.read_excel("Dublin_Ethnicity_D1_D24.xlsx"))

print("All datasets loaded successfully.")

# 2. CLEAN COMMUTE DATA

def convert_to_minutes(value):
    value = str(value).lower().strip()
    hours = re.search(r"(\d+)\s*hr", value)
    minutes = re.search(r"(\d+)\s*min", value)
    return (int(hours.group(1)) * 60 if hours else 0) + (int(minutes.group(1)) if minutes else 0)


commute["Commute Minutes"] = commute["Commute time"].apply(convert_to_minutes)
commute = commute[["Area", "University", "Commute Minutes", "Distance"]]

# 3. RENT TABLES
# Room data: double = Shared Room, single = Private Room
shared_room_rent = room_data[room_data["Room type"] == "double"].groupby("Area")["Rent"].mean()
private_room_rent = room_data[room_data["Room type"] == "single"].groupby("Area")["Rent"].mean()

rtb["Area"] = "D" + rtb["Location"].astype(str).str.extract(
    r"Dublin\s*(\d+W?)", flags=re.IGNORECASE, expand=False
).str.upper()
rtb["VALUE"] = pd.to_numeric(rtb["VALUE"], errors="coerce")
bedrooms = rtb["Number of Bedrooms"].astype(str).str.strip().str.lower()

one_bed_rent = rtb[bedrooms == "one bed"].groupby("Area")["VALUE"].mean()
two_bed_rent = rtb[bedrooms == "two bed"].groupby("Area")["VALUE"].mean()

# 4. COMBINE AREA DATA

area_data = (
    commute
    .merge(amenities, on="Area", how="left")
    .merge(census, on="Area", how="left")
    .merge(ethnicity, on="Area", how="left")
)

# 5. FACTOR DEFINITIONS
FACTORS = [
    {"name": "Rent",              "column": "Rent",              "kind": "rent"},
    {"name": "Commute",           "column": "Commute Minutes",   "kind": "commute"},
    {"name": "Supermarkets",      "column": "Supermarkets",      "kind": "count", "noun": "supermarkets"},
    {"name": "Groceries",         "column": "Groceries",         "kind": "count", "noun": "grocery shops"},
    {"name": "Restaurants",       "column": "Restaurants",       "kind": "count", "noun": "restaurants"},
    {"name": "Asian Restaurants", "column": "Asian_Restaurants", "kind": "count", "noun": "Asian restaurants"},
    {"name": "Cafes",             "column": "Cafes",             "kind": "count", "noun": "cafes"},
    {"name": "Gyms",              "column": "Gyms",              "kind": "count", "noun": "gyms"},
    {"name": "Parks",             "column": "Parks",             "kind": "count", "noun": "parks"},
    {"name": "Pharmacies",        "column": "Pharmacies",        "kind": "count", "noun": "pharmacies"},
    {"name": "Hospitals",         "column": "Hospitals",         "kind": "count", "noun": "hospitals"},
    {"name": "Shopping Malls",    "column": "Shopping_Malls",    "kind": "count", "noun": "shopping malls"},
]
FACTOR_LOOKUP = {f["name"]: f for f in FACTORS}

RATING_LABELS = {1: "Very important", 2: "Important", 3: "Slightly important", 4: "Not important"}
RATING_TO_WEIGHT = {1: 0, 2: 1, 3: 2, 4: 3}   # "Not important" contributes nothing

# 6. GET STUDENT INPUT

def ask_choice(prompt, valid):
    while True:
        answer = input(prompt).strip()
        if answer in valid:
            return answer
        print(f"Please enter one of: {', '.join(valid)}")


def ask_positive_number(prompt):
    while True:
        try:
            value = float(input(prompt).strip().replace("€", "").replace(",", ""))
            if value > 0:
                return value
        except ValueError:
            pass
        print("Please enter a positive number, e.g. 900")


print("\n==========================================")
print(" DUBLIN RESIDENTIAL AREA RECOMMENDER")
print("==========================================")

print("\nUniversity:")
print("1 = TU Dublin - Grangegorman")
ask_choice("\nEnter your university (1): ", ["1"])
university_name = "TU Dublin Grangegorman"

budget = ask_positive_number("Enter your maximum monthly accommodation budget (€): ")

ACCOMMODATION = {
    "1": ("Shared Room", shared_room_rent),
    "2": ("Private Room", private_room_rent),
    "3": ("One Bedroom Apartment", one_bed_rent),
    "4": ("Two Bedroom Apartment", two_bed_rent),
}
print("\nAccommodation options:")
for key, (label, _) in ACCOMMODATION.items():
    print(f"{key} = {label}")
accommodation_name, selected_rent = ACCOMMODATION[ask_choice("\nEnter accommodation option (1-4): ", list(ACCOMMODATION))]

print("\nRate the importance of each factor.")
for k, v in RATING_LABELS.items():
    print(f"{k} = {v.capitalize()}")

ratings = {}
for factor in FACTORS:
    ratings[factor["name"]] = int(ask_choice(f"{factor['name']}: ", ["1", "2", "3", "4"]))

weights = {name: RATING_TO_WEIGHT[r] for name, r in ratings.items()}
active_factors = [name for name, w in weights.items() if w > 0]

if not active_factors:
    raise SystemExit("\nYou rated every factor as 'Not important', so there is nothing to rank on.")

# 7. RENT + BUDGET

area_data["Rent"] = area_data["Area"].map(selected_rent)
area_data["Budget Difference"] = area_data["Rent"] - budget
area_data["Budget Status"] = np.select(
    [area_data["Rent"].isna(), area_data["Rent"] <= budget],
    ["Rent data unavailable", "Within budget"],
    default="Above budget",
)

# 8. KEEP ONLY AREAS WITH DATA FOR EVERY FACTOR THE USER CARES ABOUT

needed_columns = [FACTOR_LOOKUP[name]["column"] for name in active_factors]
needed_columns = list(dict.fromkeys(needed_columns + ["Rent"]))  # rent always needed

has_data = area_data[needed_columns].notna().all(axis=1)
candidates = area_data[has_data].copy()
excluded = area_data[~has_data].copy()


def missing_reason(row):
    missing = []
    if pd.isna(row["Rent"]):
        missing.append(f"{accommodation_name.lower()} rent")
    amenity_cols = [c for c in needed_columns if c not in ("Rent", "Commute Minutes")]
    if any(pd.isna(row[c]) for c in amenity_cols):
        missing.append("amenity counts")
    if pd.isna(row["Commute Minutes"]):
        missing.append("commute time")
    return " and ".join(missing)


if candidates.empty:
    raise SystemExit("\nNo area has complete data for your accommodation type and chosen factors.")


# 9. SCORES (0-1, higher = better), computed on the candidate set
candidates["Rent Score"] = (budget / candidates["Rent"]).clip(upper=1)

max_commute = candidates["Commute Minutes"].max()
candidates["Commute Score"] = (1 - candidates["Commute Minutes"] / max_commute).clip(lower=0) if max_commute > 0 else 1

for factor in FACTORS:
    if factor["kind"] == "count":
        col = factor["column"]
        max_value = candidates[col].max()
        candidates[f"{col} Score"] = candidates[col] / max_value if max_value > 0 else 0


def score_column(name):
    factor = FACTOR_LOOKUP[name]
    if factor["kind"] == "rent":
        return "Rent Score"
    if factor["kind"] == "commute":
        return "Commute Score"
    return f"{factor['column']} Score"


total_weight = sum(weights.values())

for name in active_factors:
    candidates[f"{name} Contribution"] = candidates[score_column(name)] * weights[name] / total_weight * 100

candidates["Match Score"] = candidates[[f"{n} Contribution" for n in active_factors]].sum(axis=1)
recommendations = candidates.sort_values("Match Score", ascending=False).reset_index(drop=True)

# 10.  RECOMMENDATION EXPLANATION

def better_or_worse(name, row, averages):

    factor = FACTOR_LOOKUP[name]
    value, avg = row[factor["column"]], averages[factor["column"]]
    if avg == 0 or abs(value - avg) / abs(avg) < 0.10:   # within 10% = similar
        return 0
    lower_is_better = factor["kind"] in ("rent", "commute")
    better = value < avg if lower_is_better else value > avg
    return 1 if better else -1


def good_phrase(name, row, ranked):

    factor = FACTOR_LOOKUP[name]
    value = row[factor["column"]]
    col = factor["column"]
    if factor["kind"] == "commute":
        if value == ranked[col].min():
            return f"is the closest to {university_name} (only {value:.0f} minutes away)"
        return f"has a short commute to {university_name} (about {value:.0f} minutes)"
    if factor["kind"] == "rent":
        if value == ranked[col].min():
            return f"is the cheapest area for a {accommodation_name.lower()} (€{value:,.0f} a month)"
        return f"has lower rent than most areas (€{value:,.0f} a month)"
    if value == ranked[col].max():
        return f"has the most {factor['noun']} ({value:.0f})"
    return f"has more {factor['noun']} than most areas ({value:.0f})"


def bad_sentence(name, row, ranked):

    factor = FACTOR_LOOKUP[name]
    value = row[factor["column"]]
    col = factor["column"]
    if factor["kind"] == "commute":
        return f"The commute is longer – about {value:.0f} minutes to {university_name}."
    if factor["kind"] == "rent":
        return f"The rent is higher than in most areas (€{value:,.0f} a month)."
    if value == ranked[col].min():
        return f"It has the fewest {factor['noun']} of all the areas ({value:.0f})."
    return f"It has fewer {factor['noun']} than most areas ({value:.0f})."


def short_name(name):
    factor = FACTOR_LOOKUP[name]
    if factor["kind"] == "commute":
        return "a shorter commute"
    if factor["kind"] == "rent":
        return "cheaper rent"
    return f"more {factor['noun']}"


def why_ahead(row_a, row_b, top_n=2):

    gaps = []
    for name in active_factors:
        gap = row_a[f"{name} Contribution"] - row_b[f"{name} Contribution"]
        if gap > 0.05:
            gaps.append((name, gap))
    gaps.sort(key=lambda x: x[1], reverse=True)
    return [short_name(name) for name, _ in gaps[:top_n]]


def join_words(items):
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


def explain(position, row, ranked, averages):
    lines = []
    n = len(ranked)

    # 1. Things this area is good at (only factors the student cares about)
    good = [name for name in active_factors if better_or_worse(name, row, averages) == 1]
    good.sort(key=lambda name: ratings[name], reverse=True)
    good = good[:3]
    if good:
        sentence = f"{row['Area']} " + join_words([good_phrase(name, row, ranked) for name in good])
        important = [name for name in good if ratings[name] >= 3]
        if len(important) == len(good):
            sentence += ", which are all things you said matter to you" if len(good) > 1 else ", which you said matters to you"
        lines.append(sentence + ".")

    # 2. Budget
    diff = row["Budget Difference"]
    if diff <= 0:
        lines.append(f"The average rent is €{row['Rent']:,.0f} a month, which is €{abs(diff):,.0f} less than your budget.")
    else:
        lines.append(f"The average rent is €{row['Rent']:,.0f} a month, which is €{diff:,.0f} more than your budget.")

    # 3. Compared with the areas next to it in the ranking
    if n > 1:
        if position < n:
            below = ranked.iloc[position]
            reasons = why_ahead(row, below)
            if reasons:
                lines.append(f"It ranked higher than {below['Area']} because it has {join_words(reasons)}.")
        if position > 1:
            above = ranked.iloc[position - 2]
            reasons = why_ahead(above, row)
            if reasons:
                lines.append(f"It ranked lower than {above['Area']} because {above['Area']} has {join_words(reasons)}.")

    # 4. Downsides on things the student said are important
    bad = [name for name in active_factors
           if ratings[name] >= 3 and better_or_worse(name, row, averages) == -1]
    bad.sort(key=lambda name: ratings[name], reverse=True)
    for i, name in enumerate(bad[:2]):
        sentence = bad_sentence(name, row, ranked)
        opener = "One downside is that " if i == 0 else "Another downside is that "
        lines.append(opener + sentence[0].lower() + sentence[1:])

    if not good and n > 1:
        lines.insert(0, "This area is about average on the things you care about.")
    return lines


def compare_word(value, average):
    if abs(value - average) < 1:
        return "about the same as"
    return "more than" if value > average else "less than"


def area_profile(row):
    lines = []
    if pd.notna(row.get("Population")) and row["Population"] > 0:
        students_pct = row["Students"] / row["Population"] * 100
        young_pct = row["Age_20_29"] / row["Population"] * 100
        dublin_students = census["Students"].sum() / census["Population"].sum() * 100
        dublin_young = census["Age_20_29"].sum() / census["Population"].sum() * 100
        lines.append(f"About {row['Population']:,.0f} people live here.")
        lines.append(f"{students_pct:.0f}% of people are students – "
                     f"{compare_word(students_pct, dublin_students)} the Dublin average ({dublin_students:.0f}%).")
        lines.append(f"{young_pct:.0f}% of people are aged 20–29 – "
                     f"{compare_word(young_pct, dublin_young)} the Dublin average ({dublin_young:.0f}%).")
    if pd.notna(row.get("Non_Irish_%")):
        groups = [c for c in ethnicity.columns if c != "Area" and pd.notna(row.get(c))]
        largest = sorted(groups, key=lambda c: row[c], reverse=True)[:2]
        group_text = " and ".join(f"{c.replace(' (%)', '')} ({row[c]:.1f}%)" for c in largest)
        lines.append(f"{row['Non_Irish_%']:.0f}% of people are not Irish. The biggest groups are {group_text}.")
    if pd.notna(row.get("Apartments_%")):
        lines.append(f"{row['Apartments_%']:.0f}% of homes are apartments.")
    return lines

# 11. DISPLAY TOP 3 RECOMMENDATION

column_averages = recommendations[[f["column"] for f in FACTORS if f["column"] in recommendations]].mean()

print("\n\n==========================================")
print(" TOP 3 RECOMMENDED AREAS")
print("==========================================")
print(f"Compared {len(recommendations)} area(s) for a {accommodation_name.lower()}, budget €{budget:,.0f}.")

explanations = {}
for position, row in recommendations.head(3).iterrows():
    position += 1
    print("\n------------------------------------------")
    print(f"#{position} {row['Area']}" + (f" ({row['LEA']})" if pd.notna(row.get("LEA")) else ""))
    print("------------------------------------------")
    print(f"Average rent:  €{row['Rent']:,.2f}  ({row['Budget Status'].lower()})")
    print(f"Commute:       {row['Commute Minutes']:.0f} min, {row['Distance']}")
    print(f"Match score:   {row['Match Score']:.1f}%")

    why_paragraph = " ".join(explain(position, row, recommendations, column_averages))
    print(f"\nWhy {row['Area']} is #{position}:")
    print(textwrap.fill(why_paragraph, width=90))

    profile = area_profile(row)
    if profile:
        print("\nAbout this area:")
        print(textwrap.fill(" ".join(profile), width=90))

    explanations[row["Area"]] = why_paragraph

['TUD G.xlsx', 'D8_Amenities_Counts.csv', 'data.md', 'D10.xlsx', 'D3_Amenities_Counts.csv', 'D6_amenities.csv', 'RTB_Clean2.csv', 'D8_amenities.csv', 'D3_amenities.csv', 'D2_Amenities_Counts.csv', 'D3.xlsx', 'Dublin_Census_Filled.xlsx', 'rent for d6 to d10.xlsx', 'D2.xlsx', 'D2_amenities.csv', 'rtb dublin rent 2025.xlsx', 'D9_Amenities_Counts.csv', 'Dublin_Ethnicity_D1_D24.xlsx', 'D10_Amenities_Counts.csv', 'D1.xlsx', 'RTB_Clean3.csv', 'D1_amenities.csv', 'D7.xlsx', 'D8.xlsx', 'RTB_Clean.csv', 'Commute_TUD_Grangegorman.csv', 'RTB_Clean1.csv', 'D6.xlsx', 'D6_Amenities_Counts.csv', 'D9.xlsx', 'D1_Amenities_Counts.csv', 'D9_amenities.csv', 'D7_amenities.csv', 'D7_Amenities_Counts.csv', 'D10_amenities.csv']
All datasets loaded successfully.

 DUBLIN RESIDENTIAL AREA RECOMMENDER

University:
1 = TU Dublin - Grangegorman

Enter your university (1): 1
Enter your maximum monthly accommodation budget (€): 1100

Accommodation options:
1 = Shared Room
2 = Private Room
3 = One Bedroom Apartment
4 

['.ipynb_checkpoints', 'D1.xlsx', 'D1_amenities.csv', 'D1_Amenities_Counts.csv', 'D2.xlsx', 'D2_amenities.csv', 'D2_Amenities_Counts.csv', 'D3.xlsx', 'D3_amenities.csv', 'D3_Amenities_Counts.csv', 'Dublin recommendation project .ipynb', 'Dublin_Census_Filled.xlsx', 'Dublin_Ethnicity_D1_D24.xlsx', 'recommendation_results.csv', 'RTB_Clean.csv', 'RTB_Clean1.csv', 'RTB_Clean2.csv', 'TUD G.xlsx', 'Untitled.ipynb']
